# Sentinel-2 L2A Automated Pipeline — CDSE

End-to-end pipeline: AOI (`.gpkg`) → scene search → selective band download → spectral indices (NDVI, SAVI, EVI, NBR, NDRE, NDWI) → validation dashboard.

All steps use the free **CDSE OData HTTP API** — no subscription needed beyond a [CDSE account](https://dataspace.copernicus.eu/).

Search for `# Change it` comments to customise paths and parameters.

In [ ]:
# 0. Authentication — CDSE OAuth2 credentials
# ─────────────────────────────────────────────────────────────────────────────
# Option A (recommended): set env vars CDSE_USER and CDSE_PASSWORD before running.
# Option B: replace the empty strings below — never commit credentials to version control.
# ─────────────────────────────────────────────────────────────────────────────
import requests
import json
import os
from pathlib import Path

CDSE_USER     = os.getenv('CDSE_USER', '')      # Change it
CDSE_PASSWORD = os.getenv('CDSE_PASSWORD', '')  # Change it

DOWNLOAD_ENDPOINT = 'https://download.dataspace.copernicus.eu'
CATALOGUE_URL     = 'https://catalogue.dataspace.copernicus.eu/odata/v1/Products'


def get_oauth2_token(user: str, password: str) -> str:
    '''
    Retrieves an OAuth2 access token from the CDSE identity server.
    Token valid for 10 minutes — refreshed automatically before each download.
    '''
    response = requests.post(
        'https://identity.dataspace.copernicus.eu'
        '/auth/realms/CDSE/protocol/openid-connect/token',
        data={
            'client_id'  : 'cdse-public',
            'username'   : user,
            'password'   : password,
            'grant_type' : 'password',
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()['access_token']


def download_band_http(
    scene_id: str,
    scene_name: str,
    band: str,
    resolution: str,
    output_path: Path,
    user: str,
    password: str,
) -> None:
    '''
    Downloads a single Sentinel-2 band file via CDSE HTTP download API.
    Refreshes the OAuth2 token before each download to avoid expiry.
    Uses streaming to avoid loading the full file into RAM.

    Args:
        scene_id:    CDSE product UUID from catalogue search.
        scene_name:  Full scene name without .SAFE for path construction.
        band:        Band identifier e.g. 'B04', 'B8A'.
        resolution:  Resolution folder e.g. 'R10m', 'R20m'.
        output_path: Destination path on local filesystem.
        user:        CDSE username (email).
        password:    CDSE password.
    '''
    # Refresh token before each download — avoids 10-min expiry on large batches
    token = get_oauth2_token(user, password)

    url = f'{DOWNLOAD_ENDPOINT}/odata/v1/Products({scene_id})/Nodes({scene_name}.SAFE)/Nodes(GRANULE)/Nodes'
    headers = {'Authorization': f'Bearer {token}'}

    # Step 1 — list GRANULE contents to find granule subfolder name
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        granule_name = response.json()['result'][0]['Id']
    except (requests.exceptions.RequestException, KeyError, IndexError) as exc:
        raise RuntimeError(f'Failed to list GRANULE for {scene_name}: {exc}')

    # Step 2 — build path to the specific band file
    band_url = (
        f'{DOWNLOAD_ENDPOINT}/odata/v1/Products({scene_id})'
        f'/Nodes({scene_name}.SAFE)'
        f'/Nodes(GRANULE)'
        f'/Nodes({granule_name})'
        f'/Nodes(IMG_DATA)'
        f'/Nodes({resolution})'
        f'/Nodes({band})'
    )

    # Step 3 — list files in resolution folder to get exact filename
    try:
        response = requests.get(band_url, headers=headers, timeout=30)
        response.raise_for_status()
        files = response.json()['result']
        band_file = next(
            (f['Id'] for f in files if band in f['Id'] and f['Id'].endswith('.tif')),
            None
        )
    except (requests.exceptions.RequestException, KeyError) as exc:
        raise RuntimeError(f'Failed to list band files for {band}: {exc}')

    if band_file is None:
        raise FileNotFoundError(
            f'Band {band} not found in {resolution} folder of {scene_name}'
        )

    # Step 4 — download the actual file with streaming
    download_url = f'{band_url}/Nodes({band_file})/$value'
    token = get_oauth2_token(user, password)   # refresh again before download

    try:
        response = requests.get(
            download_url,
            headers={'Authorization': f'Bearer {token}'},
            stream=True,
            timeout=300,   # 5-min timeout for large band files
        )
        response.raise_for_status()
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(f'Band download failed for {band_file}: {exc}')

    # Write to disk in chunks — never loads the full file into RAM
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)


# ── Run ──────────────────────────────────────────────────────────────────────
print('Requesting OAuth2 token...')
token = get_oauth2_token(CDSE_USER, CDSE_PASSWORD)
print(f'Token obtained — first 20 chars: {token[:20]}...')
print('HTTP download client ready.')

In [ ]:
# 1. AOI loading and preparation from .gpkg
import geopandas as gpd
import folium
import json
from pathlib import Path

# ── Only lines to edit ───────────────────────────────────────────────────────
GPKG_PATH  = Path('path/to/your_aoi.gpkg')  # Change it
LAYER_NAME = None                            # None to load the first layer
# ─────────────────────────────────────────────────────────────────────────────


def load_aoi(gpkg_path: Path, layer_name: str | None = None) -> dict:
    '''
    Loads a GeoPackage, reprojects to EPSG:4326, dissolves all features
    into a single polygon, and returns geometry metadata for the CDSE API.
    '''
    # Load layer
    gdf = gpd.read_file(gpkg_path, layer=layer_name)

    print('=' * 50)
    print(f'Original CRS    : {gdf.crs}')
    print(f'Feature count   : {len(gdf)}')
    print(f'Columns         : {list(gdf.columns)}')
    print(f'Geometry type   : {gdf.geom_type.unique().tolist()}')
    print('=' * 50)

    if gdf.crs is None:
        raise ValueError('The .gpkg has no CRS defined. Assign one before proceeding.')

    # Reproject to WGS84 if necessary
    if gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
        print('Reprojected to EPSG:4326')
    else:
        print('Already in EPSG:4326 — no reprojection needed')

    # Dissolve all features into a single polygon
    # Required when the .gpkg contains multiple parcels or stands
    aoi_dissolved = gdf.dissolve()
    aoi_geom = aoi_dissolved.geometry.iloc[0]

    # Compute area in hectares (requires a metric projection)
    area_ha = (
        gdf.to_crs(epsg=3857)
        .dissolve()
        .geometry
        .area
        .iloc[0]
        / 10_000
    )

    bounds = aoi_geom.bounds  # (minx, miny, maxx, maxy)

    print(f'\nBounding box (lon/lat):')
    print(f'  West  : {bounds[0]:.6f}')
    print(f'  South : {bounds[1]:.6f}')
    print(f'  East  : {bounds[2]:.6f}')
    print(f'  North : {bounds[3]:.6f}')
    print(f'  Area  : {area_ha:.2f} ha')

    # Handle MultiPolygon — use the largest sub-polygon for the WKT
    if aoi_geom.geom_type == 'MultiPolygon':
        aoi_geom = max(aoi_geom.geoms, key=lambda g: g.area)
        print('\nMultiPolygon detected — using largest polygon for WKT')

    # Build WKT for the CDSE OData API
    exterior_coords = list(aoi_geom.exterior.coords)
    wkt_coords = ', '.join(f'{x} {y}' for x, y in exterior_coords)
    wkt = f'POLYGON(({wkt_coords}))'

    return {
        'wkt'         : wkt,
        'geojson'     : json.loads(aoi_dissolved.to_json()),
        'bounds'      : bounds,
        'area_ha'     : round(area_ha, 2),
        'crs_original': str(gdf.crs),
        'n_features'  : len(gdf),
        'gdf'         : gdf,  # preserve original GDF for downstream clipping
    }


def preview_aoi(aoi_info: dict) -> folium.Map:
    '''Renders the AOI on an interactive satellite basemap.'''
    bounds  = aoi_info['bounds']
    area    = aoi_info['area_ha']
    crs_str = aoi_info['crs_original']
    n_feat  = aoi_info['n_features']

    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=13,
        tiles=(
            'https://server.arcgisonline.com/ArcGIS/rest/services'
            '/World_Imagery/MapServer/tile/{z}/{y}/{x}'
        ),
        attr='Esri World Imagery'
    )

    # AOI polygon
    folium.GeoJson(
        aoi_info['geojson'],
        name='AOI',
        style_function=lambda _: {
            'fillColor' : '#00ff88',
            'color'     : '#00cc66',
            'weight'    : 2,
            'fillOpacity': 0.2,
        }
    ).add_to(m)

    # Centre marker
    popup_html = (
        f'<b>Area:</b> {area} ha<br>'
        f'<b>Original CRS:</b> {crs_str}<br>'
        f'<b>Features:</b> {n_feat}'
    )
    folium.Marker(
        location=[center_lat, center_lon],
        popup=folium.Popup(popup_html, max_width=250),
        icon=folium.Icon(color='green', icon='leaf')
    ).add_to(m)

    folium.LayerControl().add_to(m)
    return m


# ── Run ──────────────────────────────────────────────────────────────────────
aoi_info = load_aoi(GPKG_PATH, LAYER_NAME)

print('\nWKT (first 200 characters):')
print(aoi_info['wkt'][:200] + '...')

mapa = preview_aoi(aoi_info)
display(mapa)

In [ ]:
# 2. Sentinel-2 L2A scene search via CDSE OData API
import requests
import pandas as pd
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional


@dataclass
class CDSESearchConfig:
    '''
    Configuration parameters for the CDSE OData scene search.
    All parameters are explicit — no hardcoded values anywhere.
    '''
    date_start: str          # ISO date string, e.g. '2024-06-01'
    date_end: str            # ISO date string, e.g. '2024-09-30'
    max_cloud_cover: float   # Maximum cloud cover percentage, e.g. 15.0
    max_results: int = 20    # Maximum number of scenes to retrieve from API
    collection: str = 'SENTINEL-2'                    # Copernicus collection name
    product_type: str = 'S2MSI2A'                     # L2A = atmospherically corrected
    catalogue_url: str = (                            # CDSE OData catalogue endpoint
        'https://catalogue.dataspace.copernicus.eu/odata/v1/Products'
    )
    request_timeout: int = 30                         # HTTP request timeout in seconds


class CDSESceneSearch:
    '''
    Queries the Copernicus Data Space Ecosystem OData API to find
    Sentinel-2 L2A scenes intersecting a given AOI and time window.

    Designed to receive the aoi_info dict produced by load_aoi() in Step 1.
    '''

    def __init__(self, config: CDSESearchConfig) -> None:
        self._config = config
        self._validate_config()

    def _validate_config(self) -> None:
        '''Validates date format and logical consistency of config parameters.'''
        fmt = '%Y-%m-%d'
        try:
            t_start = datetime.strptime(self._config.date_start, fmt)
            t_end   = datetime.strptime(self._config.date_end,   fmt)
        except ValueError as exc:
            raise ValueError(f'Invalid date format. Expected YYYY-MM-DD. Got: {exc}')
        if t_start >= t_end:
            raise ValueError(
                f'date_start ({self._config.date_start}) must be '
                f'before date_end ({self._config.date_end}).'
            )
        if not (0.0 <= self._config.max_cloud_cover <= 100.0):
            raise ValueError(
                f'max_cloud_cover must be in [0, 100]. '
                f'Got: {self._config.max_cloud_cover}'
            )

    def _build_odata_filter(self, wkt: str) -> str:
        '''
        Builds the OData $filter string for the CDSE catalogue query.

        Args:
            wkt: Well-Known Text polygon in EPSG:4326 representing the AOI.

        Returns:
            OData filter string ready to append to the endpoint.
        '''
        if not wkt or not wkt.startswith('POLYGON'):
            raise ValueError(f'Invalid WKT geometry. Expected POLYGON(...). Got: {wkt[:80]}')

        filters = [
            f"Collection/Name eq '{self._config.collection}'",
            (
                "Attributes/OData.CSC.StringAttribute/any("
                "att:att/Name eq 'productType' and "
                f"att/OData.CSC.StringAttribute/Value eq '{self._config.product_type}')"
            ),
            f"OData.CSC.Intersects(area=geography'SRID=4326;{wkt}')",
            f"ContentDate/Start gt {self._config.date_start}T00:00:00.000Z",
            f"ContentDate/Start lt {self._config.date_end}T23:59:59.000Z",
            (
                "Attributes/OData.CSC.DoubleAttribute/any("
                "att:att/Name eq 'cloudCover' and "
                f"att/OData.CSC.DoubleAttribute/Value le {self._config.max_cloud_cover})"
            ),
        ]
        return ' and '.join(filters)

    def _parse_cloud_cover(self, product: dict) -> float:
        '''
        Safely extracts cloud cover value from the product attributes list.

        Args:
            product: Raw product dict from CDSE OData response.

        Returns:
            Cloud cover as float, or -1.0 if the attribute is missing.
        '''
        attributes = product.get('Attributes', [])
        if not attributes:
            return -1.0
        for attr in attributes:
            if attr.get('Name') == 'cloudCover':
                try:
                    return float(attr['Value'])
                except (KeyError, TypeError, ValueError):
                    return -1.0
        return -1.0

    def _extract_tile_id(self, product_name: str) -> str:
        '''
        Extracts the MGRS tile ID from the Sentinel-2 product name.
        Example: S2B_MSIL2A_20240615T105629_N0510_R137_T30SVF_... -> T30SVF

        Args:
            product_name: Full Sentinel-2 product name string.

        Returns:
            MGRS tile identifier string, or 'UNKNOWN' if parsing fails.
        '''
        try:
            parts = product_name.split('_')
            # Tile ID is the part starting with 'T' followed by 5 alphanumeric chars
            for part in parts:
                if len(part) == 6 and part.startswith('T') and part[1:].isalnum():
                    return part
        except (IndexError, AttributeError):
            pass
        return 'UNKNOWN'

    def search(self, aoi_info: dict) -> pd.DataFrame:
        '''
        Executes the OData catalogue query and returns a tidy DataFrame
        of available scenes sorted by date ascending.

        Args:
            aoi_info: Dict produced by load_aoi() in Step 1.
                      Must contain key 'wkt' with EPSG:4326 polygon WKT.

        Returns:
            DataFrame with one row per scene, columns:
            scene_id, scene_name, tile_id, date, cloud_cover_pct, s3_path, origin
        '''
        if 'wkt' not in aoi_info or not aoi_info['wkt']:
            raise ValueError('aoi_info dict is missing the wkt key. Run load_aoi() from Step 1 first.')

        wkt = aoi_info['wkt']
        odata_filter = self._build_odata_filter(wkt)

        params = {
            '$filter'  : odata_filter,
            '$top'     : self._config.max_results,
            '$orderby' : 'ContentDate/Start asc',   # chronological order
            '$expand'  : 'Attributes',              # include cloud cover attribute
        }

        try:
            response = requests.get(
                self._config.catalogue_url,
                params=params,
                timeout=self._config.request_timeout,
            )
            response.raise_for_status()
        except requests.exceptions.Timeout:
            raise RuntimeError(
                f'CDSE catalogue request timed out after '
                f'{self._config.request_timeout}s. Check your connection.'
            )
        except requests.exceptions.HTTPError as exc:
            raise RuntimeError(
                f'CDSE catalogue returned HTTP error: {exc}. '
                f'Response body: {response.text[:300]}'
            )
        except requests.exceptions.RequestException as exc:
            raise RuntimeError(f'CDSE catalogue request failed: {exc}')

        raw_products = response.json().get('value', [])

        if not raw_products:
            print(
                f'No scenes found for the given AOI, date range '
                f'({self._config.date_start} to {self._config.date_end}), '
                f'and max cloud cover ({self._config.max_cloud_cover}%).'
            )
            return pd.DataFrame()

        # Parse each product into a clean record
        records = []
        for product in raw_products:
            name             = product.get('Name', '')
            acquisition_date = product.get('ContentDate', {}).get('Start', '')[:10]
            cloud_cover      = self._parse_cloud_cover(product)
            tile_id          = self._extract_tile_id(name)

            year, month, day = acquisition_date.split('-')
            s3_path = (
                f's3://eodata/Sentinel-2/MSI/L2A'
                f'/{year}/{month}/{day}/{name}.SAFE'
            )

            records.append({
                'scene_id'       : product.get('Id', ''),
                'scene_name'     : name,
                'tile_id'        : tile_id,
                'date'           : acquisition_date,
                'cloud_cover_pct': cloud_cover,
                's3_path'        : s3_path,
                'origin'         : 'CDSE',
            })

        df = pd.DataFrame(records).sort_values('date').reset_index(drop=True)
        return df


class CDSESceneDeduplicator:
    '''
    Removes duplicate scenes from the CDSE search results.

    Two types of duplicates are handled:
      1. Same date, same tile, different satellite (S2A vs S2B vs S2C):
         keep the scene with the lowest cloud cover.
      2. Same date, same tile, same satellite, different processing timestamp:
         keep the most recently processed version.
    '''

    def __init__(self, scenes_df: pd.DataFrame) -> None:
        if scenes_df.empty:
            raise ValueError('Input DataFrame is empty. Run CDSESceneSearch.search() first.')
        required_cols = {'date', 'tile_id', 'cloud_cover_pct', 'scene_name', 's3_path'}
        missing = required_cols - set(scenes_df.columns)
        if missing:
            raise ValueError(f'Input DataFrame missing columns: {missing}')
        self._df = scenes_df.copy()

    def _extract_processing_timestamp(self, scene_name: str) -> str:
        '''
        Extracts the processing timestamp from a Sentinel-2 product name.
        The processing timestamp is the last underscore-separated token before .SAFE.
        '''
        try:
            return scene_name.replace('.SAFE', '').split('_')[-1]
        except (IndexError, AttributeError):
            return ''

    def _extract_satellite(self, scene_name: str) -> str:
        '''Extracts the satellite identifier, e.g. S2A_MSIL2A_... -> S2A.'''
        try:
            return scene_name.split('_')[0]
        except (IndexError, AttributeError):
            return 'UNKNOWN'

    def deduplicate(self) -> pd.DataFrame:
        '''
        Applies both deduplication strategies and returns a clean DataFrame
        with one scene per date per tile — always the best available.
        '''
        df = self._df.copy()

        df['satellite']     = df['scene_name'].apply(self._extract_satellite)
        df['processing_ts'] = df['scene_name'].apply(self._extract_processing_timestamp)

        # Step 1 — resolve duplicate processing runs:
        # same date + tile + satellite -> keep most recent processing timestamp
        df = (
            df.sort_values('processing_ts', ascending=False)
            .drop_duplicates(subset=['date', 'tile_id', 'satellite'], keep='first')
            .reset_index(drop=True)
        )

        # Step 2 — resolve multi-satellite same-day coverage:
        # same date + tile, different satellite -> keep lowest cloud cover
        df = (
            df.sort_values('cloud_cover_pct', ascending=True)
            .drop_duplicates(subset=['date', 'tile_id'], keep='first')
            .reset_index(drop=True)
        )

        df = df.drop(columns=['satellite', 'processing_ts'])
        return df.sort_values('date').reset_index(drop=True)


def summarise_deduplicated(original_df: pd.DataFrame, clean_df: pd.DataFrame) -> None:
    '''Prints a before/after comparison summary of the deduplication step.'''
    removed = len(original_df) - len(clean_df)
    print('=' * 60)
    print(f'Scenes before deduplication : {len(original_df)}')
    print(f'Scenes after deduplication  : {len(clean_df)}')
    print(f'Duplicates removed          : {removed}')
    print(f'Date range                  : {clean_df["date"].min()} to {clean_df["date"].max()}')
    print(f'Avg cloud cover             : {clean_df["cloud_cover_pct"].mean():.2f}%')
    print('=' * 60)
    print('\nClean scene list:')
    print(
        clean_df[['date', 'tile_id', 'cloud_cover_pct', 'scene_name']]
        .to_string(index=True)
    )


def summarise_results(df: pd.DataFrame) -> None:
    '''Prints a human-readable summary of the scene search results.'''
    if df.empty:
        print('No results to summarise.')
        return

    print('=' * 60)
    print(f'Total scenes found  : {len(df)}')
    print(f'Date range          : {df["date"].min()} to {df["date"].max()}')
    print(f'Unique tiles        : {sorted(df["tile_id"].unique().tolist())}')
    print(f'Avg cloud cover     : {df["cloud_cover_pct"].mean():.1f}%')
    print(f'Min cloud cover     : {df["cloud_cover_pct"].min():.1f}%')
    print('=' * 60)

    n_tiles = df['tile_id'].nunique()
    if n_tiles > 1:
        print(f'WARNING: AOI spans {n_tiles} MGRS tiles. Step 3 will handle merge automatically.')

    print('\nAvailable scenes:')
    print(
        df[['date', 'tile_id', 'cloud_cover_pct', 'scene_name']]
        .to_string(index=True)
    )


# ── Run ──────────────────────────────────────────────────────────────────────
config = CDSESearchConfig(
    date_start      = '2024-06-01',   # Change it — start of temporal window
    date_end        = '2024-06-30',   # Change it — end of temporal window
    max_cloud_cover = 15.0,           # Change it — discard scenes above this threshold (%)
    max_results     = 20,             # Change it — maximum scenes to retrieve
)
searcher  = CDSESceneSearch(config)
scenes_df = searcher.search(aoi_info)
summarise_results(scenes_df)

# Deduplicate and keep best scene per date
deduplicator = CDSESceneDeduplicator(scenes_df)
clean_df     = deduplicator.deduplicate()
summarise_deduplicated(scenes_df, clean_df)

In [ ]:
# 3. Band download from CDSE HTTP API — multi-tile aware
import requests
import re
import time
from pathlib import Path
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from typing import Optional
import pandas as pd


@dataclass
class BandDownloadConfig:
    '''
    Configuration for Sentinel-2 band download via CDSE HTTP API.
    All parameters are explicit — no hardcoded values.
    '''
    output_dir: Path             # root directory where bands will be saved
    cdse_user: str               # CDSE account email
    cdse_password: str           # CDSE account password
    n_threads: int = 1           # parallel band downloads — keep at 1 to avoid HTTP 429
    download_endpoint: str = (   # CDSE HTTP download base URL
        'https://download.dataspace.copernicus.eu'
    )
    bands: tuple = (             # bands required for NDVI, SAVI, EVI, NBR, NDRE, NDWI
        ('R10m', 'B02'),         # Blue  — EVI
        ('R10m', 'B03'),         # Green — NDWI
        ('R10m', 'B04'),         # Red   — NDVI, SAVI, EVI
        ('R20m', 'B05'),         # Red Edge 1 (705 nm) — NDRE
        ('R20m', 'B8A'),         # NIR narrow — all indices
        ('R20m', 'B11'),         # SWIR1
        ('R20m', 'B12'),         # SWIR2 — NBR
    )
    request_timeout: int = 300   # seconds


class CDSEHTTPBandDownloader:
    '''
    Downloads Sentinel-2 L2A bands from CDSE via HTTP OData API.
    Multi-tile aware — groups scenes by date and downloads all tiles per date.
    Uses OAuth2 token refreshed before every request.
    '''

    def __init__(self, config: BandDownloadConfig) -> None:
        self._config = config
        self._write_lock = Lock()
        self._config.output_dir.mkdir(parents=True, exist_ok=True)

    def _get_token(self) -> str:
        '''Requests a fresh OAuth2 token — called before every HTTP request.'''
        response = requests.post(
            'https://identity.dataspace.copernicus.eu'
            '/auth/realms/CDSE/protocol/openid-connect/token',
            data={
                'client_id'  : 'cdse-public',
                'username'   : self._config.cdse_user,
                'password'   : self._config.cdse_password,
                'grant_type' : 'password',
            },
            timeout=30,
        )
        response.raise_for_status()
        return response.json()['access_token']

    def _odata_get(self, url: str) -> dict:
        '''Authenticated GET to CDSE OData API with fresh token.'''
        token = self._get_token()
        response = requests.get(
            url,
            headers={'Authorization': f'Bearer {token}'},
            timeout=30,
        )
        response.raise_for_status()
        return response.json()

    def _find_granule_name(self, scene_id: str, scene_name: str) -> str:
        '''Resolves the granule subfolder name inside the SAFE structure.'''
        url = (
            f'{self._config.download_endpoint}/odata/v1'
            f'/Products({scene_id})/Nodes({scene_name}.SAFE)/Nodes(GRANULE)/Nodes'
        )
        data = self._odata_get(url)
        results = data.get('result', [])
        if not results:
            raise RuntimeError(f'No granule found for scene {scene_name}')
        return results[0]['Id']

    def _find_band_filename(
        self,
        scene_id: str,
        scene_name: str,
        granule_name: str,
        resolution: str,
        band: str,
    ) -> str:
        '''Resolves the exact filename of a band inside the resolution folder.'''
        url = (
            f'{self._config.download_endpoint}/odata/v1'
            f'/Products({scene_id})'
            f'/Nodes({scene_name}.SAFE)'
            f'/Nodes(GRANULE)'
            f'/Nodes({granule_name})'
            f'/Nodes(IMG_DATA)'
            f'/Nodes({resolution})/Nodes'
        )
        data  = self._odata_get(url)
        files = data.get('result', [])

        for f in files:
            if re.search(rf'_{band}_\d+m\.(tif|jp2)$', f['Id'], re.IGNORECASE):
                return f['Id']

        raise FileNotFoundError(
            f'Band {band} not found in {resolution} of {scene_name}. '
            f'Available: {[f["Id"] for f in files]}'
        )

    def _download_band(
        self,
        scene_id: str,
        scene_name: str,
        granule_name: str,
        resolution: str,
        band: str,
        band_filename: str,
        local_path: Path,
    ) -> None:
        '''Downloads a single band file via HTTP streaming with retry on 429.'''
        download_url = (
            f'{self._config.download_endpoint}/odata/v1'
            f'/Products({scene_id})'
            f'/Nodes({scene_name}.SAFE)'
            f'/Nodes(GRANULE)'
            f'/Nodes({granule_name})'
            f'/Nodes(IMG_DATA)'
            f'/Nodes({resolution})'
            f'/Nodes({band_filename})/$value'
        )

        max_retries = 3
        retry_wait  = 30

        for attempt in range(1, max_retries + 1):
            token = self._get_token()
            try:
                response = requests.get(
                    download_url,
                    headers={'Authorization': f'Bearer {token}'},
                    stream=True,
                    timeout=self._config.request_timeout,
                )
                if response.status_code == 429:
                    if attempt < max_retries:
                        print(f'  429 rate limit — waiting {retry_wait}s (attempt {attempt}/{max_retries})')
                        time.sleep(retry_wait)
                        continue
                    raise RuntimeError('Rate limit persistent after retries')
                response.raise_for_status()
                break
            except requests.exceptions.RequestException as exc:
                if attempt < max_retries:
                    time.sleep(retry_wait)
                    continue
                raise RuntimeError(f'Download failed for {band_filename}: {exc}')

        with self._write_lock:
            local_path.parent.mkdir(parents=True, exist_ok=True)
            with open(local_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

    def download_scene(
        self,
        scene_id: str,
        scene_name: str,
        scene_date: str,
        tile_id: str,
    ) -> dict:
        '''
        Downloads all configured bands for a single scene.
        Output path includes tile_id to separate tiles for the same date.
        '''
        print(f'\n  Resolving granule structure for {tile_id}...')
        granule_name = self._find_granule_name(scene_id, scene_name)
        print(f'  Granule: {granule_name}')

        tasks = []
        for resolution, band in self._config.bands:
            local_path = (
                self._config.output_dir
                / scene_date
                / tile_id
                / f'{band}.jp2'
            )
            if local_path.exists():
                print(f'  SKIP: {band} ({tile_id}) already exists.')
                tasks.append((band, local_path, None, None, True))
                continue

            try:
                band_filename = self._find_band_filename(
                    scene_id, scene_name, granule_name, resolution, band
                )
                tasks.append((band, local_path, resolution, band_filename, False))
                print(f'  Found: {band} -> {band_filename}')
            except (RuntimeError, FileNotFoundError) as exc:
                print(f'  WARNING: {band} not found — {exc}')

        if not tasks:
            print(f'  WARNING: No bands resolved for {tile_id}. Skipping.')
            return {}

        n_threads  = min(self._config.n_threads, len(tasks))
        downloaded = {}

        with ThreadPoolExecutor(max_workers=n_threads) as executor:
            futures = {}
            for task in tasks:
                if task[-1] is True:
                    downloaded[task[0]] = task[1]
                    continue
                band, local_path, resolution, band_filename, _ = task
                future = executor.submit(
                    self._download_band,
                    scene_id, scene_name, granule_name,
                    resolution, band, band_filename, local_path,
                )
                futures[future] = (band, local_path)

            for future in as_completed(futures):
                band, local_path = futures[future]
                try:
                    future.result()
                    downloaded[band] = local_path
                    print(f'  OK: {band} ({tile_id}) -> {local_path}')
                except RuntimeError as exc:
                    print(f'  ERROR: {band} ({tile_id}) — {exc}')

        return downloaded

    def download_all_scenes(self, clean_df: pd.DataFrame) -> dict:
        '''
        Groups scenes by date and downloads all tiles per date.
        Returns a nested dict grouped by date for use in Step 4.
        '''
        if clean_df.empty:
            raise ValueError('clean_df is empty. Run Step 2 first.')

        required_cols = {'scene_id', 'scene_name', 'date', 'tile_id'}
        missing = required_cols - set(clean_df.columns)
        if missing:
            raise ValueError(f'clean_df missing columns: {missing}')

        dates      = clean_df['date'].unique()
        all_results = {}
        total_dates = len(dates)

        for d_idx, date in enumerate(sorted(dates)):
            date_scenes = clean_df[clean_df['date'] == date]
            tiles = date_scenes['tile_id'].tolist()
            print(f'\n[{d_idx + 1}/{total_dates}] {date} — {len(tiles)} tile(s): {tiles}')

            all_results[date] = {}

            for _, row in date_scenes.iterrows():
                scene_name = row['scene_name'].replace('.SAFE', '')
                scene_id   = row['scene_id']
                tile_id    = row['tile_id']

                try:
                    result = self.download_scene(scene_id, scene_name, date, tile_id)
                    all_results[date][tile_id] = result
                except RuntimeError as exc:
                    print(f'  ERROR: {tile_id} skipped — {exc}')
                    continue

        print('\n' + '=' * 60)
        print(f'Download complete. {len(all_results)} dates processed.')
        print(f'Output directory: {self._config.output_dir}')
        print('=' * 60)

        return all_results


# ── Run ──────────────────────────────────────────────────────────────────────
download_config = BandDownloadConfig(
    output_dir    = Path('./sentinel2_data/raw_bands'),  # Change it
    cdse_user     = CDSE_USER,                           # Change it
    cdse_password = CDSE_PASSWORD,                       # Change it
    n_threads     = 1,                                   # Keep at 1 to avoid rate limits
)

downloader = CDSEHTTPBandDownloader(download_config)
all_bands  = downloader.download_all_scenes(clean_df)

In [ ]:
# 4. Spectral index computation — NDVI, SAVI, EVI, NBR, NDRE, NDWI
# Output: one clipped float32 GeoTIFF per index per scene (multi-tile aware)
import numpy as np
import rasterio
from rasterio.mask import mask as rasterio_mask
from rasterio.merge import merge as rasterio_merge
from rasterio.enums import Resampling
from rasterio.io import MemoryFile
import geopandas as gpd
from pathlib import Path
from dataclasses import dataclass
from threading import Lock
from typing import Optional
from scipy.ndimage import zoom
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)


@dataclass
class IndexComputationConfig:
    '''
    Configuration for the spectral index computation pipeline.
    All parameters are explicit — no hardcoded values.
    '''
    bands_root: Path               # root directory containing downloaded bands
    output_root: Path              # root directory for computed index GeoTIFFs
    aoi_path: Path                 # path to AOI .gpkg for clipping
    savi_L: float = 0.5            # SAVI soil adjustment factor
    nodata_value: float = -9999.0  # nodata sentinel value
    n_threads: int = 1             # kept at 1 — CDSE rate limiting


class SpectralIndexComputer:
    '''
    Computes NDVI, SAVI, EVI, NBR, NDRE and NDWI from Sentinel-2 L2A bands.
    Multi-tile aware — merges tiles before clipping and computing indices.
    Single-tile AOIs are handled transparently without a merge step.

    Indices:
      NDVI  = (B8A - B04) / (B8A + B04)                  — general vegetation greenness
      SAVI  = ((B8A - B04) / (B8A + B04 + L)) * (1 + L)  — vegetation with soil adjustment
      EVI   = 2.5 * (B8A-B04) / (B8A+6*B04-7.5*B02+1)    — vegetation, no canopy saturation
      NBR   = (B8A - B12) / (B8A + B12)                  — burn severity, fire scars
      NDRE  = (B8A - B05) / (B8A + B05)                  — canopy chlorophyll (dense canopy)
      NDWI  = (B03 - B8A) / (B03 + B8A)                  — surface water and soil moisture
    '''

    def __init__(self, config: IndexComputationConfig) -> None:
        self._config = config
        self._write_lock = Lock()
        self._config.output_root.mkdir(parents=True, exist_ok=True)
        self._aoi_geom = self._load_aoi_geometry()

    def _load_aoi_geometry(self) -> list:
        '''Loads AOI polygon from GeoPackage as a list of GeoJSON dicts.'''
        if not self._config.aoi_path.exists():
            raise FileNotFoundError(f'AOI GeoPackage not found: {self._config.aoi_path}')
        gdf = gpd.read_file(self._config.aoi_path)
        if gdf.empty or gdf.geometry.is_empty.all():
            raise ValueError('AOI GeoPackage contains no valid geometries.')
        if gdf.crs is None:
            raise ValueError('AOI GeoPackage has no CRS defined.')
        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
        return [geom.__geo_interface__ for geom in gdf.geometry]

    def _merge_and_clip_band(self, band_paths: list[Path]) -> tuple[np.ndarray, dict]:
        '''
        Merges one or more band tiles into a single raster and clips to AOI.
        If only one tile is provided the merge step is skipped.
        Converts to float32 before masking to handle uint16 JP2 nodata.
        '''
        from shapely.geometry import shape

        for p in band_paths:
            if not p.exists():
                raise FileNotFoundError(f'Band file not found: {p}')

        if len(band_paths) == 1:
            with rasterio.open(band_paths[0]) as src:
                data    = src.read(1).astype(np.float32)
                profile = src.profile.copy()
                src_crs = src.crs
        else:
            datasets = [rasterio.open(p) for p in band_paths]
            try:
                merged, transform = rasterio_merge(datasets)
                profile = datasets[0].profile.copy()
                profile.update(
                    height    = merged.shape[1],
                    width     = merged.shape[2],
                    transform = transform,
                )
                src_crs = datasets[0].crs
                data    = merged[0].astype(np.float32)
            finally:
                for ds in datasets:
                    ds.close()

        mem_profile = profile.copy()
        mem_profile.update(dtype='float32', driver='GTiff')

        with MemoryFile() as memfile:
            with memfile.open(**mem_profile) as mem_dst:
                mem_dst.write(data, 1)

            with memfile.open() as mem_src:
                aoi_shapes = [shape(g) for g in self._aoi_geom]
                aoi_gdf = gpd.GeoDataFrame(geometry=aoi_shapes, crs='EPSG:4326')
                if src_crs.to_epsg() != 4326:
                    aoi_gdf = aoi_gdf.to_crs(src_crs)

                geoms = [g.__geo_interface__ for g in aoi_gdf.geometry]

                try:
                    clipped, clip_transform = rasterio_mask(
                        mem_src, geoms, crop=True,
                        nodata=self._config.nodata_value,
                        filled=True, all_touched=True,
                    )
                except Exception as exc:
                    raise RuntimeError(f'rasterio.mask failed: {exc}')

                out_profile = mem_src.profile.copy()
                out_profile.update(
                    driver    = 'GTiff',
                    dtype     = 'float32',
                    count     = 1,
                    nodata    = self._config.nodata_value,
                    height    = clipped.shape[1],
                    width     = clipped.shape[2],
                    transform = clip_transform,
                    compress  = 'lzw',
                    tiled     = True,
                    blockxsize= 256,
                    blockysize= 256,
                )

                return clipped[0].astype(np.float32), out_profile

    def _resample_to_target(self, arr: np.ndarray, target_shape: tuple) -> np.ndarray:
        '''Resamples array to target shape using bilinear zoom.'''
        if arr.shape == target_shape:
            return arr.astype(np.float32)
        zoom_y = target_shape[0] / arr.shape[0]
        zoom_x = target_shape[1] / arr.shape[1]
        return zoom(arr, (zoom_y, zoom_x), order=1).astype(np.float32)

    def _mask_nodata(self, *arrays: np.ndarray) -> np.ndarray:
        '''Returns boolean mask — True where any array is zero or NaN.'''
        invalid = np.zeros(arrays[0].shape, dtype=bool)
        for arr in arrays:
            invalid |= (arr == 0) | ~np.isfinite(arr)
        return invalid

    def _compute_indices(
        self,
        b02: np.ndarray,
        b03: np.ndarray,
        b04: np.ndarray,
        b05: np.ndarray,
        b8a: np.ndarray,
        b12: np.ndarray,
        nodata_mask: np.ndarray,
    ) -> dict[str, np.ndarray]:
        '''Computes NDVI, SAVI, EVI, NBR, NDRE and NDWI from surface reflectance arrays.'''
        scale = 10000.0
        blue  = b02 / scale
        green = b03 / scale
        red   = b04 / scale
        re1   = b05 / scale   # Red Edge 1 (705 nm)
        nir   = b8a / scale
        swir  = b12 / scale

        nd = self._config.nodata_value

        with np.errstate(divide='ignore', invalid='ignore'):
            # NDVI
            denom = nir + red
            ndvi = np.where(denom != 0, (nir - red) / denom, nd).astype(np.float32)

            # SAVI
            L = self._config.savi_L
            denom = nir + red + L
            savi = np.where(denom != 0, ((nir - red) / denom) * (1.0 + L), nd).astype(np.float32)

            # EVI
            denom = nir + 6.0 * red - 7.5 * blue + 1.0
            evi = np.where(denom != 0, 2.5 * (nir - red) / denom, nd).astype(np.float32)

            # NBR — burn severity, fire scars
            denom = nir + swir
            nbr = np.where(denom != 0, (nir - swir) / denom, nd).astype(np.float32)

            # NDRE — canopy chlorophyll; sensitive where NDVI saturates (dense canopy)
            # Useful for silvopastoral systems and closed-canopy forests
            denom = nir + re1
            ndre = np.where(denom != 0, (nir - re1) / denom, nd).astype(np.float32)

            # NDWI (McFeeters) — surface water and soil moisture
            # Positive = open water / wet soil; negative = vegetation / dry soil
            denom = green + nir
            ndwi = np.where(denom != 0, (green - nir) / denom, nd).astype(np.float32)

        # Apply nodata mask
        for arr in [ndvi, savi, evi, nbr, ndre, ndwi]:
            arr[nodata_mask] = nd

        # Clamp to physically valid ranges
        def clamp(arr, lo, hi):
            return np.where((arr != nd) & ((arr < lo) | (arr > hi)), nd, arr).astype(np.float32)

        ndvi = clamp(ndvi, -1.0,  1.0)
        savi = clamp(savi, -1.5,  1.5)
        evi  = clamp(evi,  -1.0,  1.0)
        nbr  = clamp(nbr,  -1.0,  1.0)
        ndre = clamp(ndre, -1.0,  1.0)
        ndwi = clamp(ndwi, -1.0,  1.0)

        return {'NDVI': ndvi, 'SAVI': savi, 'EVI': evi, 'NBR': nbr, 'NDRE': ndre, 'NDWI': ndwi}

    def _write_index(self, index_name: str, array: np.ndarray, profile: dict, output_path: Path) -> None:
        '''Writes index array to GeoTIFF, serialised via Lock.'''
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with self._write_lock:
            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(array, 1)
        print(f'  OK: {index_name} -> {output_path}')

    def process_date(self, date: str, tile_ids: list[str]) -> dict[str, Path]:
        '''
        Processes all tiles for a single date — merges tiles per band,
        clips to AOI, computes and writes all spectral indices.
        '''
        print(f'\nProcessing date: {date} — tiles: {tile_ids}')

        bands_config = [
            ('B8A', '20m reference'),     # reference grid (defines output resolution)
            ('B02', '10m Blue'),
            ('B03', '10m Green'),         # NDWI
            ('B04', '10m Red'),
            ('B05', '20m Red Edge 1'),    # NDRE
            ('B12', '20m SWIR2'),
        ]

        band_arrays = {}
        ref_profile = None

        for band, desc in bands_config:
            band_paths = [
                self._config.bands_root / date / tile / f'{band}.jp2'
                for tile in tile_ids
            ]
            print(f'  Merging {band} ({len(band_paths)} tile(s))...')
            arr, profile = self._merge_and_clip_band(band_paths)
            band_arrays[band] = arr

            if band == 'B8A':
                ref_profile = profile    # B8A defines the reference grid (20 m)

        # Resample 10m bands to the 20m B8A reference grid
        target_shape = band_arrays['B8A'].shape
        b02_arr = self._resample_to_target(band_arrays['B02'], target_shape)
        b03_arr = self._resample_to_target(band_arrays['B03'], target_shape)
        b04_arr = self._resample_to_target(band_arrays['B04'], target_shape)
        b05_arr = band_arrays['B05'].astype(np.float32)
        b8a_arr = band_arrays['B8A'].astype(np.float32)
        b12_arr = band_arrays['B12'].astype(np.float32)

        nodata_mask = self._mask_nodata(b02_arr, b03_arr, b04_arr, b05_arr, b8a_arr, b12_arr)
        indices = self._compute_indices(b02_arr, b03_arr, b04_arr, b05_arr, b8a_arr, b12_arr, nodata_mask)

        output_dir   = self._config.output_root / date
        output_paths = {}

        for index_name, index_arr in indices.items():
            output_path = output_dir / f'{index_name}.tif'
            if output_path.exists():
                print(f'  SKIP: {index_name} already exists.')
                output_paths[index_name] = output_path
                continue
            self._write_index(index_name, index_arr, ref_profile, output_path)
            output_paths[index_name] = output_path

        return output_paths

    def process_all_scenes(self, clean_df: pd.DataFrame) -> dict:
        '''Groups scenes by date and processes each date independently.'''
        if clean_df.empty:
            raise ValueError('clean_df is empty. Run Step 2 first.')

        all_results = {}
        dates = sorted(clean_df['date'].unique())
        total = len(dates)

        for d_idx, date in enumerate(dates):
            tile_ids = clean_df[clean_df['date'] == date]['tile_id'].tolist()
            print(f'\n[{d_idx + 1}/{total}] {date} — {len(tile_ids)} tile(s): {tile_ids}')

            try:
                result = self.process_date(date, tile_ids)
                all_results[date] = result
            except (FileNotFoundError, ValueError, RuntimeError) as exc:
                print(f'  ERROR: {exc} — skipping date.')
                continue

        print('\n' + '=' * 60)
        print('Index computation complete.')
        print(f'Dates processed  : {len(all_results)}')
        print(f'Output directory : {self._config.output_root}')
        print('=' * 60)

        return all_results


# ── Run ──────────────────────────────────────────────────────────────────────
index_config = IndexComputationConfig(
    bands_root  = Path('./sentinel2_data/raw_bands'),       # Change it
    output_root = Path('./sentinel2_data/index_results'),   # Change it
    aoi_path    = Path('path/to/your_aoi.gpkg'),            # Change it
    savi_L      = 0.5,  # L guide: 0 (FCC>90%), 0.25 (60-90%), 0.5 (30-60%), 0.75 (10-30%), 1 (FCC<10%)
    n_threads   = 1,
)

computer    = SpectralIndexComputer(index_config)
all_indices = computer.process_all_scenes(clean_df)

In [ ]:
# 5. Validation dashboard — spatial map + histogram per index per scene
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path


def plot_scene_indices(indices_root: Path) -> None:
    '''
    For each scene found in indices_root, plots a validation dashboard
    showing a spatial map and a histogram for each spectral index.
    '''
    index_cfg = {
        'NDVI': {'vmin': -0.1, 'vmax': 0.8,  'cmap': 'RdYlGn', 'label': 'NDVI'},
        'SAVI': {'vmin': -0.1, 'vmax': 0.8,  'cmap': 'RdYlGn', 'label': 'SAVI'},
        'EVI' : {'vmin': -0.1, 'vmax': 0.8,  'cmap': 'RdYlGn', 'label': 'EVI'},
        'NBR' : {'vmin': -0.5, 'vmax': 0.8,  'cmap': 'RdYlGn', 'label': 'NBR'},
        'NDRE': {'vmin': -0.1, 'vmax': 0.7,  'cmap': 'RdYlGn', 'label': 'NDRE'},
        'NDWI': {'vmin': -0.5, 'vmax': 0.5,  'cmap': 'RdBu',   'label': 'NDWI'},
    }

    scene_dirs = sorted([
        d for d in indices_root.rglob('*')
        if d.is_dir() and any(d.glob('NDVI.tif'))
    ])

    if not scene_dirs:
        print(f'No scenes found in {indices_root}')
        return

    for scene_dir in scene_dirs:
        scene_date = scene_dir.parent.name
        scene_name = scene_dir.name[:40]

        index_data = {}
        for index_name in ['NDVI', 'SAVI', 'EVI', 'NBR', 'NDRE', 'NDWI']:
            tif_path = scene_dir / f'{index_name}.tif'
            if not tif_path.exists():
                tif_path = indices_root / scene_dir.name / f'{index_name}.tif'
            if not tif_path.exists():
                continue
            with rasterio.open(tif_path) as src:
                arr    = src.read(1).astype(np.float32)
                nodata = src.nodata
                valid_mask = arr != nodata
                index_data[index_name] = {
                    'arr'       : arr,
                    'valid_mask': valid_mask,
                    'valid'     : arr[valid_mask],
                    'transform' : src.transform,
                    'crs'       : src.crs,
                }

        n_indices = len(index_data)
        if n_indices == 0:
            continue

        fig = plt.figure(figsize=(5 * n_indices, 8))
        fig.suptitle(
            f'Spectral Index Dashboard — {scene_date}\n{scene_name}...',
            fontsize=11, fontweight='bold', y=1.01
        )

        gs = gridspec.GridSpec(2, n_indices, figure=fig, hspace=0.45, wspace=0.35)

        for col, (index_name, data) in enumerate(index_data.items()):
            cfg   = index_cfg[index_name]
            arr   = data['arr']
            valid = data['valid']
            mask  = data['valid_mask']

            ax_map = fig.add_subplot(gs[0, col])
            display = np.where(mask, arr, np.nan)
            im = ax_map.imshow(display, cmap=cfg['cmap'], vmin=cfg['vmin'], vmax=cfg['vmax'], aspect='auto')
            plt.colorbar(im, ax=ax_map, fraction=0.046, pad=0.04)
            ax_map.set_title(cfg['label'], fontsize=11, fontweight='bold')
            ax_map.set_xticks([])
            ax_map.set_yticks([])

            ax_hist = fig.add_subplot(gs[1, col])
            ax_hist.hist(valid, bins=40, color='#2ecc71', edgecolor='#27ae60', alpha=0.85, linewidth=0.5)
            ax_hist.axvline(valid.mean(),      color='#e74c3c', lw=1.5, linestyle='--', label=f'Mean {valid.mean():.3f}')
            ax_hist.axvline(np.median(valid),  color='#3498db', lw=1.5, linestyle=':',  label=f'Median {np.median(valid):.3f}')
            ax_hist.set_xlabel(cfg['label'], fontsize=9)
            ax_hist.set_ylabel('Pixel count', fontsize=9)
            ax_hist.legend(fontsize=7, loc='upper right')
            ax_hist.tick_params(labelsize=8)

            stats_text = (
                f'Min:  {valid.min():.3f}\n'
                f'Max:  {valid.max():.3f}\n'
                f'Mean: {valid.mean():.3f}\n'
                f'Std:  {valid.std():.3f}\n'
                f'N px: {len(valid)}'
            )
            ax_hist.text(
                0.02, 0.97, stats_text,
                transform=ax_hist.transAxes, fontsize=7,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7),
            )

        plt.tight_layout()
        plt.savefig(indices_root / f'dashboard_{scene_date}.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Dashboard saved: {indices_root}/dashboard_{scene_date}.png')


# ── Run ──────────────────────────────────────────────────────────────────────
indices_root = Path('./sentinel2_data/index_results')  # Change it
plot_scene_indices(indices_root)